# 03 · Offline baselines on a time split (REAL dataset)
Popularity, recent popularity and co-visitation (`sql/03_baselines.sql`). These are incumbent-style baselines, **not** Albatross's models.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
pd.set_option('display.width', 160); pd.set_option('display.max_colwidth', 80)

In [2]:
pd.read_csv(ROOT / 'data/public/baselines/results_overall.csv')

,baseline,recall@20,ndcg@20,coverage,items_recommended,n_targets
0,(a) Global popularity,0.007489,0.002527,0.000188,20,105753
1,(b) Recent popularity (7 days),0.015517,0.005669,0.000629,67,105753
2,(c) Co-visitation + popularity fill,0.305835,0.169239,0.578247,61566,105753


## Where they fail: new and cold items, items without co-visitation history

In [3]:
seg = pd.read_csv(ROOT / 'data/public/baselines/results_by_segment.csv')
seg.pivot_table(index=['dimension', 'segment'], columns='baseline', values='recall@20').round(3)

baseline                                     (a) Global popularity  (b) Recent popularity (7 days)  (c) Co-visitation + popularity fill
dimension segment                                                                                                                      
context   1 item seen                                        0.008                           0.017                                0.335
          2+ items seen                                      0.007                           0.014                                0.284
prev      current item has co-visitation                     0.008                           0.017                                0.349
          current item has no co-visitation                  0.001                           0.008                                0.008
target    cold item (1-4)                                    0.000                           0.001                                0.067
          new item (0 in train)                              0.000                           0.006                                0.001
          warm item (5+)                                     0.009                           0.018                                0.361
user      new user                                           0.008                           0.016                                0.314
          returning user                                     0.006                           0.014                                0.265

## Co-visitation in SQL
A self-join of training sessions on themselves: every pair of different items seen in the same session counts once.

In [4]:
sql = (ROOT / 'sql/03_baselines.sql').read_text()
print(sql[sql.index('-- name: covis\n'):sql.index('-- name: covis_lists')])

-- name: covis
-- Item-to-item co-visitation from TRAINING sessions: number of sessions in which
-- both items were viewed. Sessions are capped at 30 distinct items (long sessions
-- would otherwise dominate and the pair count grows quadratically).
CREATE OR REPLACE TABLE covis AS
WITH tr AS (
    SELECT DISTINCT s.session_id, s.item_id FROM sess s JOIN sess_start USING (session_id)
    WHERE start_ts < (SELECT split_ts FROM params) AND s.ts < (SELECT split_ts FROM params)
),
capped AS (
    SELECT * FROM tr WHERE session_id IN (
        SELECT session_id FROM tr GROUP BY 1 HAVING count(*) BETWEEN 2 AND 30)
),
pairs AS (
    SELECT a.item_id AS item_a, b.item_id AS item_b, count(*) AS n_sessions
    FROM capped a JOIN capped b ON a.session_id = b.session_id AND a.item_id <> b.item_id
    GROUP BY 1, 2
)
SELECT item_a, item_b, n_sessions,
       row_number() OVER (PARTITION BY item_a ORDER BY n_sessions DESC, item_b) AS rnk
FROM pairs QUALIFY rnk <= 20;



